# Marshall Wace — Codility Prep (via Saragossa)

**Source:** recruiter (Ilyssa Alagon) description of the test ahead of booking, 2026-07-13.

- 3 questions, 2 hours, Codility (Leetcode-style)
- Auto-marked, then reviewed by a dev — **visible test cases must pass, hidden ones too**, and clean/readable code matters because a human reads it after
- 80% is the pass mark → progresses to interview
- Budget roughly 35-40 min per question, leaving a buffer to re-check edge cases

**The three questions, as described:**
1. Generate unique email addresses from names (optional middle names), de-duplicating with an appended integer
2. **The hard one** — max number of uniquely-tagged points that fit inside a circle centred at the origin
3. Reconstruct a 2-row matrix from row sums + a column-sum array, or determine it's impossible

**Protocol — same rules as `cross_ocean_mock_test.ipynb` and `PROGRESS.md`:**
1. Read the explanation for each question once, then attempt the `YOUR TURN` cells yourself before looking anywhere else.
2. Trace through the approach on paper/out loud before coding — especially Q2.
3. Run every assert cell. If one fails, read *why* before you touch the code again.
4. Solutions are at the very bottom, behind a warning. This is real interview prep, not a discovery exercise — read the explanations freely, but write the code yourself first.

In [ ]:
from collections import Counter
print('Ready.')

---
## Q1 — Unique Email Generator

**Likely shape:** you're given a list of people (first name, last name, optional middle name) and need to generate a company email for each, using some naming convention (e.g. `first.last@company.com`, inserting the middle name when present). When two people would generate the *same* email, later ones get an integer appended so every address stays unique.

**This is the same pattern as LeetCode 1487 (Making File Names Unique).** The base-name convention (whether middle names are used, whether it's `first.last` or `f.last`, dots vs no dots) will be spelled out in the actual prompt — don't guess it in advance, just make sure the *de-duplication logic* is airtight, because that's the actual algorithmic content.

**The core algorithm (this is the part worth knowing cold):**
- Keep a dict mapping `name -> next suffix to try`.
- First time you see a base name, use it as-is, record suffix `0`.
- If you've seen it before, try appending `1`, `2`, `3`... starting from the recorded suffix, until you find one that hasn't been used **at all** (not just as a base name — a previous person's *suffixed* email might already equal your candidate).
- Update the dict for both the base name (bump its suffix) and the new suffixed name (mark it used).

**The edge case that trips people up:** person A collides and becomes `john.smith1`. Then person C's own *natural* base name is `john.smith1` (e.g. last name is literally "Smith1"). Your candidate check has to catch that collision too, or you'll silently produce two identical emails. This is exactly what test 2 below checks.

In [ ]:
# Step 1 — build the base email string (adjust the convention once you see the real prompt)
# YOUR TURN
def make_email(first, last, middle=None):
    pass

assert make_email("John", "Smith") == "john.smith@company.com"
assert make_email("Jane", "Doe", "Ann") == "jane.ann.doe@company.com"
assert make_email(" John ", "SMITH") == "john.smith@company.com"  # normalise case/whitespace
print("Q1 step 1 passed")

In [ ]:
# Step 2 — the full de-duplication pass over a list of people
# people: list of dicts like {"first": "John", "last": "Smith", "middle": "Ann"}  (middle optional)
# YOUR TURN
def generate_unique_emails(people):
    pass

people = [
    {"first": "John", "last": "Smith"},
    {"first": "John", "last": "Smith"},
    {"first": "John", "last": "Smith"},
    {"first": "Jane", "middle": "Ann", "last": "Doe"},
]
out = generate_unique_emails(people)
assert out == [
    "john.smith@company.com",
    "john.smith1@company.com",
    "john.smith2@company.com",
    "jane.ann.doe@company.com",
], out

# the tricky one: a later person's NATURAL base name collides with an earlier person's SUFFIXED email
people2 = [
    {"first": "John", "last": "Smith"},    # -> john.smith
    {"first": "John", "last": "Smith"},    # collision -> john.smith1
    {"first": "John", "last": "Smith1"},   # natural base IS john.smith1 -> must not collide!
]
out2 = generate_unique_emails(people2)
assert len(out2) == len(set(out2)), out2

# no people
assert generate_unique_emails([]) == []
print("Q1 step 2 passed:", out, out2)

---
## Q2 — Unique-Tag Circle (the hard one)

**Restated precisely:** you're given a set of points, each with a letter tag (tags can repeat across different points). You choose a circle centred at the origin (any radius you like). A point is "in" the circle if its distance from the origin is within the radius. You want the **maximum number of points** you can capture in some circle such that:
- No two captured points share a tag.
- If two points share a tag *and* sit at exactly the same distance from the origin, they can never both be captured (see below).

**The key insight:** because the circle is centred at the origin, "inside the circle" is *entirely determined by distance from the origin*. That means if you sort every point by distance ascending, any valid circle corresponds to exactly a **prefix** of that sorted list — you cannot pick point #7 without also picking every point #1-6 that's closer in. There's no way to "skip" a closer point to reach a farther one.

That turns this into a single greedy sweep, closest to farthest:
- Walk outward. Keep a set of tags already captured.
- The moment you'd be forced to capture a point whose tag is already in your set, **stop** — you cannot go any further out, because any bigger circle still contains that conflicting point too. The answer is however many points you'd captured right before that.

**Why ties (same distance) need special handling:** if two points sit at *exactly* the same distance, a circle can never separate them — either both are inside or both are outside, because they're on the same circle boundary. So you must process points in **groups of equal distance**, all-or-nothing:
- If a group has two points with the *same tag as each other*, that whole group is "poisoned" — you can never safely include it (both would come in together), so you stop before it.
- If a group is internally clean but one of its tags was already used by a smaller-distance group, it's also poisoned — stop before it.
- Otherwise, take the whole group, add its tags to the used set, and keep going.

**Worked example:** points at (3,4,'A'), (5,0,'B'), (10,0,'A') — distances are 5, 5, 10.
- Group at distance 5: {A, B} — no internal duplicate, nothing used yet → capture both. Used tags: {A, B}. Count = 2.
- Group at distance 10: {A} — but 'A' is already used → poisoned, stop here.
- Answer: **2**.

**Implementation gotcha:** don't compare `sqrt(x**2 + y**2)` values for equality — floating point sqrt will occasionally make two points that are *supposed* to be at the same distance compare as unequal (or vice versa), and a hidden test case almost certainly targets exactly this. Compare **squared distance** (`x**2 + y**2`), which is exact integer arithmetic if the coordinates are integers. Only take the sqrt if you actually need to print/return an actual distance value (you don't, here).

In [ ]:
# Step 1 — squared distance (avoid sqrt entirely for comparisons)
# YOUR TURN
def sq_dist(x, y):
    pass

assert sq_dist(3, 4) == 25
assert sq_dist(0, 0) == 0
assert sq_dist(-3, -4) == 25
print("Q2 step 1 passed")

In [ ]:
# Step 2 — group points into equal-distance clusters, in ascending distance order
# Return a list of groups, each group a list of tags, e.g. [['A'], ['A','B'], ['C']]
# YOUR TURN
def group_by_distance(points):
    # points: list of (x, y, tag)
    pass

groups = group_by_distance([(3,4,'A'), (5,0,'B'), (10,0,'A')])
assert groups == [['A', 'B'], ['A']], groups

groups2 = group_by_distance([(1,0,'X'), (2,0,'Y'), (3,0,'Z')])
assert groups2 == [['X'], ['Y'], ['Z']], groups2

assert group_by_distance([]) == []
print("Q2 step 2 passed")

In [ ]:
# Step 3 — put it together: the full greedy sweep
# YOUR TURN
def max_points_in_circle(points):
    # points: list of (x, y, tag)
    pass

# all unique tags, all different distances -> take everything
assert max_points_in_circle([(1,0,'A'), (2,0,'B'), (3,0,'C')]) == 3

# duplicate tag farther out -> stop before it
assert max_points_in_circle([(1,0,'A'), (2,0,'B'), (5,0,'A')]) == 2

# two points, same tag, same exact distance -> poisoned immediately, answer 0
assert max_points_in_circle([(3,4,'A'), (5,0,'A')]) == 0

# two points, same distance, different tags -> both fit
assert max_points_in_circle([(3,4,'A'), (5,0,'B')]) == 2

# clean group, then a later point re-uses one of that group's tags
assert max_points_in_circle([(3,4,'A'), (5,0,'B'), (10,0,'A')]) == 2

# empty input
assert max_points_in_circle([]) == 0

# single point
assert max_points_in_circle([(1,1,'A')]) == 1

# degenerate: two points at the origin itself, same tag
assert max_points_in_circle([(0,0,'A'), (0,0,'A')]) == 0

print("Q2 step 3 passed — all edge cases covered")

---
## Q3 — Matrix Reconstruction from Row/Column Sums

**This is LeetCode 1253 ("Reconstruct a 2-Row Binary Matrix").** You're given `upper` (row 0's total sum), `lower` (row 1's total sum), and `colsum` (an array where `colsum[i]` is the sum of column `i` across both rows — so each entry is 0, 1, or 2). Build a valid 0/1 matrix with exactly those row and column sums, or determine it's impossible.

**Greedy, column by column:**
- `colsum[i] == 2` → both rows must be 1 in this column (only way to sum to 2 with 0/1 values). Deduct 1 from both `upper` and `lower`.
- `colsum[i] == 0` → both rows are 0. Nothing to deduct.
- `colsum[i] == 1` → exactly one row is 1. Give it to whichever of `upper`/`lower` currently has more *remaining* budget — this is the greedy part. (Doesn't matter which you pick when they're tied, but you must pick the larger one when they differ, otherwise you can strand the smaller one with more to place than columns left to place it in.)

**Feasibility checks — these are what "impossible" actually means:**
- Any `colsum[i]` outside `{0, 1, 2}` is invalid on its face.
- If `upper` or `lower` ever goes negative while processing, it's impossible.
- After processing every column, if `upper` and `lower` aren't *both* exactly 0, it's impossible — you had leftover budget with no columns left to place it in.

**Edge cases worth testing explicitly:** all-zero input, a `colsum` that's individually valid per-entry but whose total doesn't match `upper + lower`, and the case where `upper`/`lower` start at 0 but `colsum` demands something be placed.

In [ ]:
# YOUR TURN
def reconstruct_matrix(upper, lower, colsum):
    # return [row0, row1] as lists of 0/1, or None if impossible
    pass

# feasible, straightforward
res = reconstruct_matrix(2, 1, [1,1,1])
assert res is not None
assert [a+b for a,b in zip(res[0], res[1])] == [1,1,1]
assert sum(res[0]) == 2 and sum(res[1]) == 1

# feasible, includes a 2 and a 0
res2 = reconstruct_matrix(2, 2, [1,1,1,1])
assert res2 is not None
assert sum(res2[0]) == 2 and sum(res2[1]) == 2
assert [a+b for a,b in zip(res2[0], res2[1])] == [1,1,1,1]

# impossible: totals don't add up
assert reconstruct_matrix(1, 1, [2,2]) is None

# impossible: invalid colsum entry (not 0/1/2)
assert reconstruct_matrix(1, 1, [3]) is None

# edge: everything zero
assert reconstruct_matrix(0, 0, [0,0,0]) == [[0,0,0],[0,0,0]]

# edge: budget with no room to place it
assert reconstruct_matrix(0, 0, [1]) is None

print("Q3 passed — all edge cases covered")

---
## General Codility Strategy (80% pass mark, auto + human reviewed)

- **Write the obvious brute force first if you're stuck**, get *something* passing the visible cases, then optimise. Partial credit exists — a working O(n²) beats a broken O(n log n).
- **Re-read the constraints** before choosing an approach — they tell you the expected complexity (e.g. n ≤ 10⁴ is fine for O(n log n), n ≤ 10⁹ means you need something closer to O(n) or O(log n)).
- **Name things properly.** A dev reads this after — `d`, `tmp`, `x2` don't cut it when the grader is a person, not just Codility's checker.
- **Test the sample I/O the instant you have something running**, don't wait until the function is "finished" in your head.
- **Handle empty/singleton input explicitly** — it's the single most common hidden-test miss.
- For Q2 specifically: **squared distance, not sqrt**, for any equality comparison. This is very likely a deliberate hidden-test trap given how the problem is described.

---
## Solutions

**Stop.** Only scroll past this line once you've genuinely attempted every `YOUR TURN` cell above. If you're reading this before attempting, at minimum trace through Q2's worked example on paper first — reading the answer without reconstructing the reasoning won't hold up if the dev reviewing your submission asks you to explain your approach.

In [ ]:
# --- Q1 ---
def make_email(first, last, middle=None):
    parts = [first.strip().lower()]
    if middle:
        parts.append(middle.strip().lower())
    parts.append(last.strip().lower())
    return ".".join(parts) + "@company.com"

def generate_unique_emails(people):
    seen = {}
    result = []
    for p in people:
        base = make_email(p["first"], p["last"], p.get("middle"))
        local, domain = base.split("@")
        if base not in seen:
            seen[base] = 0
            result.append(base)
        else:
            k = seen[base] + 1
            candidate = f"{local}{k}@{domain}"
            while candidate in seen:
                k += 1
                candidate = f"{local}{k}@{domain}"
            seen[base] = k
            seen[candidate] = 0
            result.append(candidate)
    return result

In [ ]:
# --- Q2 ---
def sq_dist(x, y):
    return x**2 + y**2

def group_by_distance(points):
    pts = sorted(points, key=lambda p: sq_dist(p[0], p[1]))
    groups = []
    i = 0
    n = len(pts)
    while i < n:
        d = sq_dist(pts[i][0], pts[i][1])
        j = i
        tags = []
        while j < n and sq_dist(pts[j][0], pts[j][1]) == d:
            tags.append(pts[j][2])
            j += 1
        groups.append(tags)
        i = j
    return groups

def max_points_in_circle(points):
    used_tags = set()
    count = 0
    for group in group_by_distance(points):
        if len(set(group)) != len(group):          # internal duplicate tag in this group
            break
        if any(t in used_tags for t in group):      # conflicts with an already-captured tag
            break
        used_tags.update(group)
        count += len(group)
    return count

In [ ]:
# --- Q3 ---
def reconstruct_matrix(upper, lower, colsum):
    n = len(colsum)
    row0 = [0]*n
    row1 = [0]*n
    for i, c in enumerate(colsum):
        if c not in (0, 1, 2):
            return None
        if c == 2:
            row0[i] = 1
            row1[i] = 1
            upper -= 1
            lower -= 1
        elif c == 1:
            if upper >= lower:
                row0[i] = 1
                upper -= 1
            else:
                row1[i] = 1
                lower -= 1
        if upper < 0 or lower < 0:
            return None
    if upper != 0 or lower != 0:
        return None
    return [row0, row1]